# Подготовка выгрузок из МАРКЕРА (отчеты «Результаты» и «Цены»)



Получаю чистые таблицы по новому строительству школ (без капремонта и реконструкции)

На выходе будет 3 файла в `data_processed/`:

- **`marker_results_main_clean.csv`** — 1 строка = 1 закупка/публикация (то, по чему скачиваю документы и беру стоимость для расчёта руб./место, руб./$м^2$)
- **`marker_results_participants_clean.csv`** — строки уровня «участник/статус» (поставщик, ИНН, допуск, победитель, цена). Это нужно для гипотез про конкуренцию
- **`marker_prices_all_clean.csv`** — детализация «Цены» (строки товаров/позиций)

In [1]:
from pathlib import Path
import re
import pandas as pd
import openpyxl

BASE_DIR = Path("/Users/arinazajceva/Desktop/диплом")
DATA_XLSX = BASE_DIR / "data строительство"
OUT_DIR = BASE_DIR / "data_processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_XLSX, OUT_DIR


(PosixPath('/Users/arinazajceva/Desktop/диплом/data строительство'),
 PosixPath('/Users/arinazajceva/Desktop/диплом/data_processed'))

In [2]:
files = [Path('/Users/arinazajceva/Desktop/диплом/data строительство/результаты.xlsx'),
        Path('/Users/arinazajceva/Desktop/диплом/data строительство/цены.xlsx')]
files

[PosixPath('/Users/arinazajceva/Desktop/диплом/data строительство/результаты.xlsx'),
 PosixPath('/Users/arinazajceva/Desktop/диплом/data строительство/цены.xlsx')]

Что делаю:

1. Считываю выгрузки из МАРКЕРА: отчеты **«Результаты»** и **«Цены»**
2. Привожу названия колонок к нормальному виду, вытаскиваем ссылки из Excel `HYPERLINK()`
3. Убираем технические/повторяющиеся строки (дубликаты): отдельно для «главной» таблицы и для «участников»
4. Фильтрую выборку (оставляю только новое строительство)
5. Сохраняю итоговые файлы в `data_processed/`

In [ ]:
# приводим название колонки к нормальному виду: убираем лишние пробелы и переносы
def normalize_colname(s):
    s = str(s)
    s = re.sub(r"\s+", " ", s.strip())
    s = s.replace("\n", " ")
    return s


# в excel ссылки хранятся как формула HYPERLINK("url", "текст"), pandas их не читает, поэтому разбираю вручную через openpyxl
def parse_hyperlink(v):
    if not isinstance(v, str):
        return None, None
    s = v.strip()
    if not s.upper().startswith("=HYPERLINK("):
        return None, None

    start = s.find("(")
    end = s.rfind(")")
    if start == -1 or end == -1 or end <= start:
        return None, None

    inside = s[start + 1 : end]

    args = []
    i = 0
    n = len(inside)
    while i < n:
        if inside[i] != '"':
            i += 1
            continue
        i += 1
        buf = []
        while i < n:
            ch = inside[i]
            if ch == '"':
                if i + 1 < n and inside[i + 1] == '"':
                    buf.append('"')
                    i += 2
                    continue
                i += 1
                break
            buf.append(ch)
            i += 1
        args.append("".join(buf))
        if len(args) >= 2:
            break

    url = args[0] if len(args) >= 1 else None
    text = args[1] if len(args) >= 2 else None
    return url, text


def get_hyperlink_column(path, header_name, n_rows):
    wb = openpyxl.load_workbook(path, data_only=False)
    ws = wb.active

    headers = [normalize_colname(ws.cell(row=1, column=i).value) for i in range(1, ws.max_column + 1)]
    if header_name not in headers:
        wb.close()
        return [None] * n_rows, [None] * n_rows

    col_idx = headers.index(header_name) + 1

    urls, texts = [], []
    for r in range(2, 2 + n_rows):
        v = ws.cell(row=r, column=col_idx).value
        url, text = parse_hyperlink(v)
        urls.append(url)
        texts.append(text)

    wb.close()
    return urls, texts

In [ ]:
def read_xlsx(path):
    df = pd.read_excel(path)
    df.columns = [normalize_colname(c) for c in df.columns]

    urls, texts = get_hyperlink_column(path, "Наименование публикации", len(df))
    df["publication_url"] = urls
    df["publication_name_full"] = texts
    df["Наименование публикации"] = df["publication_name_full"].fillna(df["Наименование публикации"])

    urls, texts = get_hyperlink_column(path, "Ссылка на источник", len(df))
    df["source_url"] = urls
    df["source_text"] = texts

    df["_source_file"] = path.name
    return df

sample_results = read_xlsx(files[0])
sample_prices = read_xlsx(files[1])
cols = [c for c in ["Наименование публикации", "publication_url", "Ссылка на источник", "source_url"] if c in sample_results.columns]
(sample_results.shape, sample_prices.shape, sample_results[cols].head(3))


((282, 39),
 (547, 30),
                              Наименование публикации                                    publication_url    Ссылка на источник                                         source_url
 0  Выполнение работ по объекту: "Строительство зд...  https://analytics.marker-zakupki.ru/Card/Lot/1...  Госзакупки 44ФЗ/94ФЗ  https://zakupki.gov.ru/epz/order/notice/ok20/v...
 1  Выполнение строительно-монтажных и прочих рабо...  https://analytics.marker-zakupki.ru/Card/Lot/1...  Госзакупки 44ФЗ/94ФЗ  https://zakupki.gov.ru/epz/order/notice/ea20/v...
 2  Выполнение строительно-монтажных и прочих рабо...  https://analytics.marker-zakupki.ru/Card/Lot/1...  Госзакупки 44ФЗ/94ФЗ  https://zakupki.gov.ru/epz/order/notice/ea20/v...)

Смотрю, какие колонки получились после загрузки

In [5]:
sample_results.columns

Index(['Уровень', 'Заказчик', 'ИНН заказчика', 'Стоимость (руб.) Заказчик', 'Реестровый номер публикации', 'Идентификационный код закупки', 'Сфера деятельности', 'Наименование публикации',
       'Регион поставки', 'Город поставки', 'Дата публикации', 'Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ',
       'Дата начала подачи заявок/Дата начала исполнения контракта / Дата публикации ППГ', 'Дата окончания проведения торгов', 'Поставщик', 'ИНН поставщика', 'Победитель', 'Статус допуска',
       'Стоимость (руб.) Поставщик', 'Снижение на торгах,%', 'Форма публикации', 'Тип торгов', 'Торговая площадка', 'Электронные торги', 'Обеспечение заявки (руб.)', 'Обеспечение заявки, %',
       'Обеспечение контракта (руб.)', 'Обеспечение контракта, %', 'Банковское \ казначейское сопровождение', 'Источник финансирования', 'Нацрежим', 'Ссылка на источник', 'Цвет', 'Комментарий',
       'publication_url', 'publication_name_full', 'so

In [6]:
sample_prices.columns

Index(['Уровень', 'Реестровый номер публикации', 'Номер лота', 'Наименование публикации', 'Регион поставки', 'Город поставки', 'Сфера деятельности', 'Валюта', 'Стоимость в валюте',
       'Наименование товара', 'Кол-во', 'Ед. изм.', 'Цена за ед. в валюте', 'Цена за ед. (руб.)', 'Стоимость (руб.)', 'Роль компании', 'Компания', 'ИНН', 'Дата публикации',
       'Дата начала подачи заявок', 'Ссылка на источник', 'Реестровый номер контракта', 'Наименование контракта', 'Цвет', 'Комментарий', 'publication_url', 'publication_name_full', 'source_url',
       'source_text', '_source_file'],
      dtype='object')

Убираем дубликаты по набору колонок (если указан prefer_col — при дублях оставляем строку, где это поле заполнено (обычно цена))

In [ ]:
def dedupe_simple(df, subset, prefer_col=None):
    if not subset:
        return df

    out = df.copy()

    if prefer_col and prefer_col in out.columns:
        pref = pd.to_numeric(out[prefer_col], errors="coerce")
        out["__has"] = pref.notna().astype(int)
        out["__pref"] = pref
        out = out.sort_values(subset + ["__has", "__pref"])
        out = out.drop_duplicates(subset=subset, keep="last")
        out = out.drop(columns=["__has", "__pref"])
    else:
        out = out.drop_duplicates(subset=subset, keep="first")

    return out

In [ ]:
p = files[0]
tmp = read_xlsx(p)
print(tmp.shape)

results_all = tmp.rename(
    columns={
        "Наименование публикации": "publication_name",
        "Реестровый номер публикации": "registry_number",
        "Идентификационный код закупки": "purchase_code",
        "Снижение на торгах,%": "discount_pct",
        "Стоимость (руб.) Поставщик": "supplier_price_rub",
        "Стоимость (руб.) Заказчик": "customer_price_rub",
        "Форма публикации": "publication_form",
    }
)

(282, 39)


Объединяем все отчеты "Цены" в один датафрейм

In [ ]:
p = files[1]
tmp = read_xlsx(p)
print(tmp.shape)

prices_all = tmp.rename(
    columns={
        "Реестровый номер публикации": "registry_number_publication",
        "Реестровый номер контракта": "registry_number_contract",
        "Наименование публикации": "publication_name",
        "Регион поставки": "region",
        "Город поставки": "city",
        "ОКПД2": "okpd2",
        "Номер лота": "lot_number",
    }
)

(547, 30)


In [10]:
results_all.head()

,Уровень,Заказчик,ИНН заказчика,customer_price_rub,registry_number,purchase_code,Сфера деятельности,publication_name,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок/Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,supplier_price_rub,discount_pct,publication_form,Тип торгов,Торговая площадка,Электронные торги,Обеспечение заявки (руб.),"Обеспечение заявки, %",Обеспечение контракта (руб.),"Обеспечение контракта, %",Банковское \ казначейское сопровождение,Источник финансирования,Нацрежим,Ссылка на источник,Цвет,Комментарий,publication_url,publication_name_full,source_url,source_text,_source_file
0,1,АДМИНИСТРАЦИЯ ВОЛОДАРСКОГО МУНИЦИПАЛЬНОГО ОКРУГА,5.214002e+09,9.853270e+08,132300007425000261,25-35214001770521401001-0216-555-4120-414,[ОКПД2 41.20] Здания и работы по возведению зд...,"Выполнение работ по объекту: ""Строительство зд...",Нижегородская область,Володарский район,46021.434676,46049.416667,46021.434676,46052.0,NaN,NaN,NaN,NaN,NaN,NaN,Торговая процедура,Конкурс открытый,Фабрикант,504ФЗ,4926635.15,0.01,9853270.30,0.01,Требуется казначейское сопровождение контракта,Неизвестно,Нет,Госзакупки 44ФЗ/94ФЗ,NaN,NaN,https://analytics.marker-zakupki.ru/Card/Lot/1...,"Выполнение работ по объекту: ""Строительство зд...",https://zakupki.gov.ru/epz/order/notice/ok20/v...,Госзакупки 44ФЗ/94ФЗ,результаты.xlsx
1,1,"ГБУ ""ГЛАВСТРОЙ РТ""",1.655396e+09,6.138783e+07,311500000925000175,25-21655395979165501001-0237-001-4120-407,[ОКПД2 41.20] Здания и работы по возведению зд...,Выполнение строительно-монтажных и прочих рабо...,Республика Татарстан (Татарстан),Республика Татарстан (Татарстан),45996.574097,46006.458333,45996.574097,46008.0,Несколько поставщиков,NaN,NaN,NaN,NaN,NaN,Торговая процедура,Аукцион электронный,ZAKAZRF,504ФЗ,3069391.50,0.05,18416349.00,0.30,Неизвестно,Неизвестно,Нет,Госзакупки 44ФЗ/94ФЗ,NaN,NaN,https://analytics.marker-zakupki.ru/Card/Lot/1...,Выполнение строительно-монтажных и прочих рабо...,https://zakupki.gov.ru/epz/order/notice/ea20/v...,Госзакупки 44ФЗ/94ФЗ,результаты.xlsx
2,2,"ГБУ ""ГЛАВСТРОЙ РТ""",1.655396e+09,6.138783e+07,311500000925000175,25-21655395979165501001-0237-001-4120-407,[ОКПД2 41.20] Здания и работы по возведению зд...,Выполнение строительно-монтажных и прочих рабо...,Республика Татарстан (Татарстан),Республика Татарстан (Татарстан),45996.574097,46006.458333,45996.574097,46008.0,Неизвестно,NaN,Победитель,Допущен,61080890.85,0.005,Торговая процедура,NaN,NaN,NaN,3069391.50,0.05,18416349.00,0.30,Неизвестно,NaN,Нет,Госзакупки 44ФЗ/94ФЗ,NaN,NaN,https://analytics.marker-zakupki.ru/Card/Lot/1...,Выполнение строительно-монтажных и прочих рабо...,https://zakupki.gov.ru/epz/order/notice/ea20/v...,Госзакупки 44ФЗ/94ФЗ,результаты.xlsx
3,2,"ГБУ ""ГЛАВСТРОЙ РТ""",1.655396e+09,6.138783e+07,311500000925000175,25-21655395979165501001-0237-001-4120-407,[ОКПД2 41.20] Здания и работы по возведению зд...,Выполнение строительно-монтажных и прочих рабо...,Республика Татарстан (Татарстан),Республика Татарстан (Татарстан),45996.574097,46006.458333,45996.574097,46008.0,Неизвестно,NaN,NaN,Допущен,61080890.85,0.005,Торговая процедура,NaN,NaN,NaN,3069391.50,0.05,18416349.00,0.30,Неизвестно,NaN,Нет,Госзакупки 44ФЗ/94ФЗ,NaN,NaN,https://analytics.marker-zakupki.ru/Card/Lot/1...,Выполнение строительно-монтажных и прочих рабо...,https://zakupki.gov.ru/epz/order/notice/ea20/v...,Госзакупки 44ФЗ/94ФЗ,результаты.xlsx
4,1,КУ ЧР СЛУЖБА ЕДИНОГО ЗАКАЗЧИКА,2.130135e+09,4.401121e+08,815500000525015229,25-22130135250213001001-0033-001-4120-414,[ОКПД2 41.20] Здания и работы по возведению зд...,Строительство пристроя на 400 ученических мест...,Чувашская Республика - Чувашия,Вурнарский район,45980.687616,45992.375000,45980.687616,45994.0,Несколько поставщиков,NaN,NaN,NaN,NaN,NaN,Торговая процедура,Аукцион электронный,ЕЭТП (Р

В выгрузке МАРКЕРА одна закупка может повторяться в нескольких строках — нужно убрать дубли и разделить на две таблицы

Из одной таблицы делаю 2:
1) results_main: 1 строка на закупку (для расчета удельной стоимости)
2) results_participants: строки с данными по участникам (для анализа конкуренции)

In [ ]:
results_key = ["registry_number", "purchase_code"]

dup_rows = results_all.duplicated(subset=results_key).sum()
dup_groups = results_all.groupby(results_key).size().gt(1).sum()
print(f"results_main: ключ={results_key}; дублей строк={dup_rows}; групп с дублями={dup_groups}")

results_main: ключ=['registry_number', 'purchase_code']; дублей строк=125; групп с дублями=50


In [ ]:
main = results_all.copy()
main["price"] = pd.to_numeric(main["supplier_price_rub"], errors="coerce")
main["has_price"] = main["price"].notna().astype(int)
main["level"] = pd.to_numeric(main.get("Уровень"), errors="coerce")

win = main.get("Победитель")
if win is None:
    main["is_winner"] = 0
else:
    s = win.fillna("").astype(str).str.lower()
    main["is_winner"] = s.str.contains("побед").astype(int)

# при нескольких строках на закупку оставляем ту, где есть цена победителя
main = main.sort_values(
    results_key + ["has_price", "is_winner", "level", "price"],
    ascending=[True, True, False, False, False, True],
)
results_main = main.drop_duplicates(subset=results_key, keep="first").drop(
    columns=["price", "has_price", "is_winner", "level"],
    errors="ignore",
)

Таблица участников: берем строки, где есть статус допуска, победитель или цена поставщика

In [ ]:
_part = results_all.copy()
_part["_price"] = pd.to_numeric(_part["supplier_price_rub"], errors="coerce")

has_price = _part["_price"].notna()
has_admission = _part["Статус допуска"].notna()
has_winner = _part["Победитель"].notna()

results_participants_raw = _part.loc[has_price | has_admission | has_winner].copy()

participants_key = [
    "registry_number",
    "purchase_code",
    "Победитель",
    "Статус допуска",
    "supplier_price_rub",
    "ИНН поставщика",
    "Поставщик",
]
participants_key = [c for c in participants_key if c in results_participants_raw.columns]

dup_rows = results_participants_raw.duplicated(subset=participants_key).sum()
dup_groups = results_participants_raw.groupby(participants_key).size().gt(1).sum()
print(f"results_participants: ключ={participants_key}; дублей строк={dup_rows}; групп с дублями={dup_groups}")

results_participants = dedupe_simple(results_participants_raw, subset=participants_key)
results_participants = results_participants.drop(columns=["_price"], errors="ignore")

print("results_all:", results_all.shape)
print("results_main:", results_main.shape)
print("results_participants:", results_participants.shape)


results_main: ключ=['registry_number', 'purchase_code']; дублей строк=125; групп с дублями=50
results_participants: ключ=['registry_number', 'purchase_code', 'Победитель', 'Статус допуска', 'supplier_price_rub', 'ИНН поставщика', 'Поставщик']; дублей строк=6; групп с дублями=0
results_all: (282, 39)
results_main: (157, 39)
results_participants: (183, 39)


Таблица цен

In [ ]:
prices_key = [
    "registry_number_publication",
    "lot_number",
    "Наименование товара",
    "Кол-во",
    "Ед. изм.",
    "Цена за ед. в валюте",
    "Цена за ед. (руб.)",
    "Стоимость в валюте",
    "Стоимость (руб.)",
    "Роль компании",
    "Компания",
    "ИНН",
]

dup_rows = prices_all.duplicated(subset=prices_key).sum()
dup_groups = prices_all.groupby(prices_key).size().gt(1).sum()
print(f"prices: ключ={prices_key}; дублей строк={dup_rows}; групп с дублями={dup_groups}")

prices_all_dedup = dedupe_simple(prices_all, subset=prices_key)

print("prices:", prices_all.shape, "->", prices_all_dedup.shape)

prices: ключ=['registry_number_publication', 'lot_number', 'Наименование товара', 'Кол-во', 'Ед. изм.', 'Цена за ед. в валюте', 'Цена за ед. (руб.)', 'Стоимость в валюте', 'Стоимость (руб.)', 'Роль компании', 'Компания', 'ИНН']; дублей строк=25; групп с дублями=0
prices: (547, 30) -> (522, 30)


Посмотрим на итоговую таблицу — должна остаться 1 строка на закупку

In [ ]:
results_main.head()

,Уровень,Заказчик,ИНН заказчика,customer_price_rub,registry_number,purchase_code,Сфера деятельности,publication_name,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок/Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,supplier_price_rub,discount_pct,publication_form,Тип торгов,Торговая площадка,Электронные торги,Обеспечение заявки (руб.),"Обеспечение заявки, %",Обеспечение контракта (руб.),"Обеспечение контракта, %",Банковское \ казначейское сопровождение,Источник финансирования,Нацрежим,Ссылка на источник,Цвет,Комментарий,publication_url,publication_name_full,source_url,source_text,_source_file
186,1,"МБОУ ""СЕВЕРНАЯ СОШ №2""",5.645002e+09,5.134048e+05,6448182,NaN,[ОКПД2 41.20] Здания и работы по возведению зд...,Капитальное строительство центра образования е...,Оренбургская область,Северный район,45146.256655,45146.503472,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Торговая процедура,Запрос предложений,РТС-тендер,Неизвестно,NaN,NaN,NaN,NaN,NaN,Неизвестно,Нет,РТС-ТЕНДЕР 44-ФЗ,NaN,NaN,https://analytics.marker-zakupki.ru/Card/Lot/1...,Капитальное строительство центра образования е...,https://market.rts-tender.ru/zapros/6448182,РТС-ТЕНДЕР 44-ФЗ,результаты.xlsx
47,1,С ПЛОЩАДКИ РТС ТЕНДЕР,NaN,2.500000e+05,8815987,NaN,[ОКПД2 41.10] Документация проектная для строи...,Выполнение работ по оценке технического состоя...,Оренбургская область,Оренбургский район,45768.273021,45770.378472,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Торговая процедура,Запрос предложений,РТС-тендер,Неизвестно,NaN,NaN,NaN,NaN,NaN,Неизвестно,Нет,РТС-ТЕНДЕР 44-ФЗ,NaN,NaN,https://analytics.marker-zakupki.ru/Card/Lot/1...,Выполнение работ по оценке технического состоя...,https://market.rts-tender.ru/search/sell/88159...,РТС-ТЕНДЕР 44-ФЗ,результаты.xlsx
262,1,АДМИНИСТРАЦИЯ МР ХАЙБУЛЛИНСКИЙ РАЙОН РБ,2.480052e+08,3.332589e+08,101500000322000125,22-30248005212024801001-0031-001-4120-414,[ОКПД2 41.20] Здания и работы по возведению зд...,Школа на 550 мест с интернатом на 140 мест в с...,Республика Башкортостан,Республика Башкортостан,44692.723206,44708.375000,44692.639965,44713.000000,Неизвестно,NaN,NaN,Допущен,333258880.0,0.0,Торговая процедура,Конкурс открытый,Фабрикант,504ФЗ,3332588.80,0.01,33325888.00,0.1,Требуется банковское сопровождение контракта,Бюджет Республики Башкортостан; Бюджет муницип...,Нет,Госзакупки 44ФЗ/94ФЗ,NaN,NaN,https://analytics.marker-zakupki.ru/Card/Lot/1...,Школа на 550 мест с интернатом на 140 мест в с...,https://zakupki.gov.ru/epz/order/notice/ok20/v...,Госзакупки 44ФЗ/94ФЗ,результаты.xlsx
234,1,ГКУ УКС РБ,2.781765e+08,6.772817e+08,101500000322000183,22-20278176470027601001-0348-001-4120-414,[ОКПД2 41.20] Здания и работы по возведению зд...,"Выполнение строительно-монтажных, пусконаладоч...",Республика Башкортостан,Абзелиловский район,44789.731620,44813.291667,44789.648403,44817.916667,NaN,NaN,NaN,NaN,NaN,NaN,Торговая процедура,Конкурс открытый,Фабрикант,504ФЗ,33864085.54,0.05,67728171.09,0.1,Требуется банковское и казначейское сопровожде...,Неизвестно,Нет,Госзакупки 44ФЗ/94ФЗ,NaN,NaN,https://analytics.marker-zakupki.ru/Card/Lot/1...,"Выполнение строительно-монтажных, пусконаладоч...",https://zakupki.gov.ru/epz/order/notice/ok20/v...,Госзакупки 44ФЗ/94ФЗ,результаты.xlsx
232,1,ГКУ УКС РБ,2.781765e+08,7.459681e+08,101500000322000257,22-20278176470027601001-0348-002-4120-414,[ОКПД2 41.20] Здания и работы по возведению зд...,"Выполнение строительно-монтажных, пусконаладоч...",Республика Башкортостан,Абзелиловский район,44839.378160,44855.291667,44839.295012,44859.916667,NaN,NaN,NaN,NaN,NaN,NaN,Торговая процедура,Конкурс открытый,Фабрикант,504ФЗ,37298406.28,0.05,74596812.57,0.1,Требуется банковское и казначейское сопровожде...,Неизвестно,Нет,Госзакупки 44ФЗ/94ФЗ,NaN,NaN,https://analytics.marker-zakupki.ru/Card/Lot/1...,"Выполнение строительно-мон

In [ ]:
# Главная таблица (1 строка на закупку)
results_main_out_csv = OUT_DIR / "marker_results_main_clean.csv"

# Таблица участников (много строк на закупку)
results_participants_out_csv = OUT_DIR / "marker_results_participants_clean.csv"

# Цены (товары/позиции)
prices_out_csv = OUT_DIR / "marker_prices_all_clean.csv"

results_main.to_csv(results_main_out_csv, index=False)
results_participants.to_csv(results_participants_out_csv, index=False)
prices_all_dedup.to_csv(prices_out_csv, index=False)

results_main_out_csv, results_participants_out_csv, prices_out_csv

(PosixPath('/Users/arinazajceva/Desktop/диплом/data_processed/marker_results_main_clean.csv'),
 PosixPath('/Users/arinazajceva/Desktop/диплом/data_processed/marker_results_participants_clean.csv'),
 PosixPath('/Users/arinazajceva/Desktop/диплом/data_processed/marker_prices_all_clean.csv'))